# DeepGuard — Benchmark Run All

One-click Colab pipeline for the DF40 benchmark. It uses the modern Colab environment, never installs the obsolete `lir` package, keeps DF40 on Google Drive, and separates detector evaluation from DeepGuard-LR calibration.


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys, shutil, json, os
drive.mount('/content/drive', force_remount=False)
ROOT=Path('/content/drive/MyDrive/DeepGuard')
DF40=ROOT/'datasets/DF40'
DF40_CODE=Path('/content/DF40')
WEIGHTS=ROOT/'models/df40/xception.pth'
JSON_DIR=ROOT/'preprocessing/dataset_json'
RESULTS=ROOT/'benchmark/df40'
for p in [RESULTS, ROOT/'manifests']: p.mkdir(parents=True,exist_ok=True)
print('GPU:',subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null') or 'NO GPU')
print('Drive free GiB:',round(shutil.disk_usage('/content/drive').free/1024**3,1))
print('DF40:',DF40)


In [ ]:
# Get the official DF40 code only; data stay on Drive.
if not DF40_CODE.exists():
    subprocess.run(['git','clone','https://github.com/YZY-stack/DF40.git',str(DF40_CODE)],check=True)
print('DF40 code ready:',DF40_CODE)


In [ ]:
# Preflight. This cell never starts a large run when something is missing.
checks={
 'DF40 data':DF40.exists(),
 'dataset_json':JSON_DIR.exists(),
 'Xception weights':WEIGHTS.exists(),
 'DF40 test.py':(DF40_CODE/'training/test.py').exists(),
 'Xception config':(DF40_CODE/'training/config/detector/xception.yaml').exists(),
}
for k,v in checks.items(): print(f'{k:20} {"PASS" if v else "MISSING"}')
if not all(checks.values()): print('STOP: fix the missing items above; no detector has been started.')


In [ ]:
# Copy official dataset JSONs into the DF40 code location without copying the image data.
TARGET_JSON=DF40_CODE/'preprocessing/dataset_json'
if JSON_DIR.exists():
    TARGET_JSON.mkdir(parents=True,exist_ok=True)
    for p in JSON_DIR.glob('*.json'):
        q=TARGET_JSON/p.name
        if not q.exists(): q.symlink_to(p)
print('JSON count:',len(list(TARGET_JSON.glob('*.json'))))


In [ ]:
# Protocol 3 Xception run. Only runs when all prerequisites are present.
runner=Path('/content/deepguard-forensic-lr')
if not runner.exists(): subprocess.run(['git','clone','https://github.com/geradts/deepguard-forensic-lr.git',str(runner)],check=True)
cmd=[sys.executable,str(runner/'scripts/run_df40_large_benchmark.py'),'--drive-root',str(ROOT),'--df40-root',str(DF40_CODE),'--weights',str(WEIGHTS),'--protocol','p3','--detector','xception','--dry-run']
print(' '.join(map(str,cmd)));
if all(checks.values()): subprocess.run(cmd,check=True)


## Real Xception run

After the dry-run succeeds, change `--dry-run` to a real run. The runner writes an audit manifest, checkpoint SHA-256, log and exit code to Drive.


In [ ]:
# REAL RUN — leave commented until the dry-run above says PASS.
# cmd=[sys.executable,str(runner/'scripts/run_df40_large_benchmark.py'),'--drive-root',str(ROOT),'--df40-root',str(DF40_CODE),'--weights',str(WEIGHTS),'--protocol','p3','--detector','xception']
# subprocess.run(cmd,check=True)
print('Xception real-run cell is intentionally guarded. Uncomment after preflight + dry-run PASS.')


In [ ]:
# Fusion stage: expects detector_scores_train.csv and detector_scores_test.csv.
train=RESULTS/'detector_scores_train.csv'; test=RESULTS/'detector_scores_test.csv'
if train.exists() and test.exists():
    out=RESULTS/'deepguard_fused_scores.csv'; metrics=RESULTS/'deepguard_metrics.json'
    cmd=[sys.executable,str(runner/'scripts/fuse_detector_scores.py'),'--train-csv',str(train),'--test-csv',str(test),'--out-csv',str(out),'--metrics-json',str(metrics)]
    subprocess.run(cmd,check=True)
else:
    print('LR fusion waiting for Xception/FTCN detector score tables.')


## FTCN

FTCN is deliberately a second stage. We will add its official implementation/checkpoint after Xception is validated, then produce the joint `Xception + FTCN + forensic features → DeepGuard-LR` result. The external case videos remain outside development/calibration.